**Imports**

In [ ]:
import yaml
import json
import pandas as pd
import numpy as np
import xlsxwriter
from math import floor
import xlwings as xw

**Helper Functions**

In [ ]:
def conventional_round(n, places=0):
    multiplier = 10 ** places
    return floor(n*multiplier + 0.5) / multiplier

def add_rows(df, col):
    """given a dataframe (df) and scalar column name (col),
    this will insert a blank row after each unique value in col

    Args:
        df: pd.Dataframe to add rows to
        col: scalar (str) 

    Returns:
        pd.Dataframe
    """
    df = df.reset_index(drop = True)
    m = df[col].ne(df[col].shift(-1)) 
    df = pd.concat([df,pd.DataFrame(data=None, index = m.index[m] + 0.5, columns = df.columns)])
    return df.sort_index().reset_index(drop=True).iloc[:-1]

**Read Data**

In [ ]:
root = "J:\\EUA\\CRA\\Collateral Review\\Volatility Factor\\HIST\\2025_lv_effective\\LendingValue\\python_inputs\\" #enter path to current analysis 'python_inputs' folder
fname = "lending_value_python_inputs_2025.xlsx" #enter current analysis 'python_inputs' filename
fpath = root + fname

MAPPING = pd.read_excel(fpath, sheet_name = "mapping", dtype = {"COLLATERAL_PRIORITY": float}) #mapping

CFACTOR = pd.read_excel(fpath, sheet_name = "ca_factors", 
                   dtype = {"CREDIT_A": float, "CREDIT_B": float, "MODEL_PRICE": float, "MODEL_UPB": float,
                            "LIQUIDATION_PREP": float, "LIQUIDATION_SALE": float}) #credit admin factors
                            
VOLFACTOR = pd.read_excel(fpath,sheet_name = "vol_factors", dtype = {"10D": float, "84D": float, "252D": float}) #vol factors
VOLFACTOR = VOLFACTOR.set_index("VOLATILITY_FACTOR_ID").to_dict()

LVP = pd.read_excel(fpath, sheet_name = "present_lv", dtype = {"LV_A": float, "LV_B": float}) #present lvs

WSFORMAT = {'border': 1, 'border_color': 'black'}

HEADFORMAT = {'bold': True}

**Function: Combine Data**

In [ ]:
def assemble_data():
    return (
        MAPPING
        .merge(CFACTOR, on = "CREDIT_ADMIN_FACTOR_ID", how="left")
        .merge(LVP, on = ["COLLATERAL_GROUP_CODE", "COLLATERAL_ACE_CODE", "COLLATERAL_TABLE", "FHLB_RATING"], how="left")
        .assign(VOLATILITY_A = lambda _df: np.where(_df["COLLATERAL_TABLE"]=="Securities", 
                                                    _df["VOLATILITY_FACTOR_ID"].map(VOLFACTOR["10D"]),
                                                    _df["VOLATILITY_FACTOR_ID"].map(VOLFACTOR["84D"])),
                VOLATILITY_B = lambda _df: _df["VOLATILITY_FACTOR_ID"].map(VOLFACTOR["252D"]),
                PROPOSED_LV_CL_A_RAW = lambda _df: np.minimum(100 - (_df[["CREDIT_A", "CREDIT_B", "MODEL_PRICE", "MODEL_UPB", "LIQUIDATION_PREP", 
                                                            "LIQUIDATION_SALE", "VOLATILITY_A"]].sum(axis=1)).apply(lambda x: conventional_round(x)),99), #Class A
                PROPOSED_LV_CL_B_RAW = lambda _df: np.minimum(100 - (_df[["CREDIT_A", "CREDIT_B", "MODEL_PRICE", "MODEL_UPB", "LIQUIDATION_PREP",
                                                            "LIQUIDATION_SALE", "VOLATILITY_B"]].sum(axis=1)).apply(lambda x: conventional_round(x)),99), #Class B
                PROPOSED_LV_CL_A = lambda _df: np.select([_df["COLLATERAL_ACE_CODE"].isin(["CDSXXX", "ONDXXX"]),
                                                        _df["CREDIT_ADMIN_FACTOR_ID"] == "not_eligible",
                                                        _df["NLL_CODE"].astype(str) == "0",
                                                        _df["NLL_CODE"].astype(str) != "0"],
                                                        [100, 0, _df["PROPOSED_LV_CL_A_RAW"], _df["PROPOSED_LV_CL_A_RAW"]-10]),
                PROPOSED_LV_CL_B = lambda _df: np.select([_df["COLLATERAL_ACE_CODE"].isin(["CDSXXX", "ONDXXX"]),
                                                        _df["CREDIT_ADMIN_FACTOR_ID"] == "not_eligible",
                                                        _df["NLL_CODE"].astype(str) == "0",
                                                        _df["NLL_CODE"].astype(str) != "0"],
                                                        [100, 0, _df["PROPOSED_LV_CL_B_RAW"], _df["PROPOSED_LV_CL_B_RAW"]-10]),                           
                CHANGE_CL_A = lambda _df: _df["PROPOSED_LV_CL_A"] - _df["LV_A"],
                CHANGE_CL_B = lambda _df: _df["PROPOSED_LV_CL_B"] - _df["LV_B"]
                )
        .rename({"LV_A": "PRESENT_LV_CL_A", "LV_B": "PRESENT_LV_CL_B"}, axis=1)
        .sort_values(["COLLATERAL_PRIORITY", "COLLATERAL_ACE_CODE", "FHLB_RATING"], ascending=[True, True, False])
    )

**Assemble Data**

In [ ]:
df = assemble_data()

**Management LV Override**

In [ ]:
#class A
#df.loc[(df["COLLATERAL_ACE_CODE"] == "MFXXXX") & (df["COLLATERAL_TABLE"] == "Limited"), "PROPOSED_LV_CL_A"] = 70 #uncomment to apply management override of LV
#df.loc[(df["COLLATERAL_ACE_CODE"] == "MFXXXX") & (df["COLLATERAL_TABLE"] == "QCD"), "PROPOSED_LV_CL_A"] = 70 #uncomment to apply management override of LV

#class B
#df.loc[(df["COLLATERAL_ACE_CODE"] == "MFXXXX") & (df["COLLATERAL_TABLE"] == "Limited"), "PROPOSED_LV_CL_B"] = 67 #uncomment to apply management override of LV
#df.loc[(df["COLLATERAL_ACE_CODE"] == "MFXXXX") & (df["COLLATERAL_TABLE"] == "QCD"), "PROPOSED_LV_CL_B"] = 67 #uncomment to apply management override of LV

**Exhibit A**

In [ ]:
writer = pd.ExcelWriter("output\Exhibit_A_Detail.xlsx", engine="xlsxwriter") #Exhibit A - xlsx excel object
writerCSO = pd.ExcelWriter("output\Exhibit_A_CSO.xlsx", engine="xlsxwriter") #Exhibit A (FS) - xlsx excel object

wb = writer.book #Exhibit A - workbook object
wbCSO = writerCSO.book #Exhibit A (FS) - workbook object

for tbl in ["Exhibit_A", "Expanded", "Limited", "QCD", "Warehouse", "Securities"]:
    #Step 1: Filter dataframe
    if tbl == "Exhibit_A":
        dft = df.copy()
    else:
        dft = df[df["COLLATERAL_TABLE"]==tbl].copy()

    #Step 2: add rows; set index (for grouped excel output)
    dft = add_rows(dft, "COLLATERAL_GROUP_CODE")
    dft = dft.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

    #Step 3a: Create Exhibit A
    cols = ["COLLATERAL_ACE_CODE", "ACE_CODE_DESC", "COLLATERAL_TABLE", "FHLB_RATING", 
            "CREDIT_ADMIN_FACTOR_ID", "VOLATILITY_FACTOR_ID", "CREDIT_A", "CREDIT_B", 
            "MODEL_PRICE", "MODEL_UPB", "LIQUIDATION_PREP", "LIQUIDATION_SALE", 
            "VOLATILITY_A", "VOLATILITY_B", "PROPOSED_LV_CL_A", "PROPOSED_LV_CL_B", 
            "PRESENT_LV_CL_A", "PRESENT_LV_CL_B", "CHANGE_CL_A", "CHANGE_CL_B"]
    dft[cols].to_excel(writer, sheet_name=tbl) #dft to xlsxwriter object
    ws = writer.sheets[tbl] #worksheet object
    ws.set_column('A:W', None, wb.add_format(WSFORMAT)) #set column format
    ws.set_default_row(15) #set row height
    ws.set_header('&L&A') #set header
    wb.add_format(HEADFORMAT) #set header format

    if tbl == "Exhibit_A":
        ws.set_paper(17)
        ws.set_landscape()
        ws.center_horizontally()
        ws.set_margins(left = 0.25, right = 0.25, top = 0.50, bottom = 0.50)
        ws.repeat_rows(0)
        ws.fit_to_pages(1, 5)

    #Step 3b: Create Exhibit A (CSO)
    cols = ["COLLATERAL_ACE_CODE", "ACE_CODE_DESC", "COLLATERAL_TABLE", 
            "FHLB_RATING", "CREDIT_ADMIN_FACTOR_ID", "VOLATILITY_FACTOR_ID", 
            "PROPOSED_LV_CL_A", "PROPOSED_LV_CL_B"]
    dft[cols].to_excel(writerCSO, sheet_name=tbl)
    ws = writerCSO.sheets[tbl] #worksheet object
    ws.set_column('A:K', None, wbCSO.add_format(WSFORMAT)) #set column format
    ws.set_default_row(15) #set row height
    ws.set_header('&L&A') #set header
    wbCSO.add_format(HEADFORMAT) #set header format

    if tbl == "Exhibit_A":
        ws.set_paper(1)
        ws.set_landscape()
        ws.center_horizontally()
        ws.set_margins(left = 0.25, right = 0.25, top = 0.50, bottom = 0.50)
        ws.repeat_rows(0)
        ws.fit_to_pages(1, 7)

writer.close() #Exhibit A - xlsx excel object
writerCSO.close() #Exhibit A - xlsx excel object

#Exhibit A - format and add non-public sensitivity label
wb_a = xw.Book("output\Exhibit_A_Detail.xlsx")

for tbl in ["Exhibit_A", "Expanded", "Limited", "QCD", "Warehouse", "Securities"]:
    ws = wb_a.sheets[tbl]
    a_range = ws.used_range.address 
    ws[a_range].font.name = "Calibri"
    ws[a_range].font.size = 10
    ws[a_range].columns.autofit()
    ws[a_range].api.HorizontalAlignment = xw.constants.HAlign.xlHAlignCenter
    ws[a_range].api.VerticalAlignment = xw.constants.VAlign.xlVAlignCenter 
    
    if tbl == "Exhibit_A":
        ws.api.PageSetup.PrintArea = a_range    

labelinfo = wb_a.api.SensitivityLabel.CreateLabelInfo()
labelinfo.AssignmentMethod = 2
labelinfo.Justification = 'init'
labelinfo.LabelId = "7212d102-6a13-4378-b0e2-fdaee0e76765"
wb_a.api.SensitivityLabel.SetLabel(labelinfo, labelinfo)

wb_a.save()
wb_a.close()

#Exhibit A_CSO - format and add non-public sensitivity label
wb_a_cso = xw.Book("output\Exhibit_A_CSO.xlsx")

for tbl in ["Exhibit_A", "Expanded", "Limited", "QCD", "Warehouse", "Securities"]:
    ws = wb_a_cso.sheets[tbl]
    a_range = ws.used_range.address 
    ws[a_range].font.name = "Calibri"
    ws[a_range].font.size = 10
    ws[a_range].columns.autofit()
    ws[a_range].api.HorizontalAlignment = xw.constants.HAlign.xlHAlignCenter
    ws[a_range].api.VerticalAlignment = xw.constants.VAlign.xlVAlignCenter

    if tbl == "Exhibit_A":
        ws.api.PageSetup.PrintArea = a_range   

labelinfo = wb_a_cso.api.SensitivityLabel.CreateLabelInfo()
labelinfo.AssignmentMethod = 2
labelinfo.Justification = 'init'
labelinfo.LabelId = "7212d102-6a13-4378-b0e2-fdaee0e76765"
wb_a_cso.api.SensitivityLabel.SetLabel(labelinfo, labelinfo)

wb_a_cso.save()
wb_a_cso.close()

**Loans Cheat Sheet**

In [ ]:
writer = pd.ExcelWriter("output\Loan_Cheat_Sheet_CSO.xlsx", engine="xlsxwriter") #Loan Cheat Sheet - xlsx excel object

wb = writer.book #Cheat Sheet (Loan) - workbook object 

for tbl in ["Loan_Cheat_Sheet", "Expanded", "Limited", "QCD", "Warehouse"]:
    #Step 1: Filter dataframe
    if tbl == "Loan_Cheat_Sheet":
        dft = df.loc[(df["COLLATERAL_TABLE"] != "Securities")].copy()
    else:
        dft = df[df["COLLATERAL_TABLE"] == tbl].copy()

    #Step 2: add rows; set index (for grouped excel output)
    dft = add_rows(dft, "COLLATERAL_GROUP_CODE")
    dft = dft.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

    #Step 3: Create cheat sheet
    cols = ["COLLATERAL_ACE_CODE", "ACE_CODE_DESC", "COLLATERAL_TABLE", 
        "PROPOSED_LV_CL_A", "PROPOSED_LV_CL_B", "CAP", "CDFI"]
    dft[cols].to_excel(writer, sheet_name = tbl)
    ws = writer.sheets[tbl] #worksheet object
    ws.set_column('A:J', None, wb.add_format(WSFORMAT)) #set column format
    ws.set_default_row(15) #set row height
    ws.set_header('&L&A') #set header
    wb.add_format(HEADFORMAT) #set header format

    if tbl == "Loan_Cheat_Sheet":
        ws.set_paper(1)
        ws.set_landscape()
        ws.center_horizontally()
        ws.set_margins(left = 0.25, right = 0.25, top = 0.50, bottom = 0.50)
        ws.repeat_rows(0)
        ws.fit_to_pages(1, 7)

writer.close()

#Loan Cheat Sheet - format and add non-public sensitivity label
wb_l = xw.Book("output\Loan_Cheat_Sheet_CSO.xlsx")

for tbl in ["Loan_Cheat_Sheet", "Expanded", "Limited", "QCD", "Warehouse"]:
    ws = wb_l.sheets[tbl]
    a_range = ws.used_range.address 
    ws[a_range].font.name = "Calibri"
    ws[a_range].font.size = 10
    ws[a_range].columns.autofit()
    ws[a_range].api.HorizontalAlignment = xw.constants.HAlign.xlHAlignCenter
    ws[a_range].api.VerticalAlignment = xw.constants.VAlign.xlVAlignCenter

    if tbl == "Loan_Cheat_Sheet":
        ws.api.PageSetup.PrintArea = a_range

labelinfo = wb_l.api.SensitivityLabel.CreateLabelInfo()
labelinfo.AssignmentMethod = 2
labelinfo.Justification = 'init'
labelinfo.LabelId = "7212d102-6a13-4378-b0e2-fdaee0e76765"
wb_l.api.SensitivityLabel.SetLabel(labelinfo, labelinfo)

wb_l.save()
wb_l.close()

**Securities Cheat Sheet**

In [ ]:
import pandas.io.formats.excel
pandas.io.formats.excel.header_style = None 

writer = pd.ExcelWriter("output\Security_Cheat_Sheet_CSO.xlsx", engine="xlsxwriter") #Security Cheat Sheet - xlsx excel object

wb = writer.book #Cheat Sheet (Securities) - workbook object

#Step 1: Filter dataframe
tbl = "Security_Cheat_Sheet"
dft = df.loc[(df["COLLATERAL_TABLE"] == "Securities")].copy()

#Step 2: add rows; set index (for grouped excel output)
dft = add_rows(dft, "COLLATERAL_GROUP_CODE")
dft = dft.set_index(["COLLATERAL_GROUP_CODE", "ACE_GROUP_DESC", "COLLATERAL_PRIORITY"])

#Step 3: Create cheat sheet
dft["BLANKET_CL_A"] = dft["PROPOSED_LV_CL_A"] #reassign col 
dft["SPECIFIC_CL_A"] = dft["PROPOSED_LV_CL_A"] #reassign col 
dft["SPECIFIC_CL_B"] = dft["PROPOSED_LV_CL_B"] #reassign col 

cols =  ["COLLATERAL_ACE_CODE", "ACE_CODE_DESC", "FHLB_RATING", "BLANKET_CL_A", "SPECIFIC_CL_A", "SPECIFIC_CL_B", "MDA_REVIEW_REQ", "CASE_BY_CASE_FEE"]

dft[cols].to_excel(writer, sheet_name = tbl, index=True) 
ws = writer.sheets[tbl] #worksheet object
ws.set_column("A:K", None, wb.add_format(WSFORMAT)) #set column format
ws.set_default_row(15) #set row height
ws.set_header('&L&A') #set header
wb.add_format(HEADFORMAT) #set header format

ws.set_paper(1)
ws.set_landscape()
ws.center_horizontally()
ws.set_margins(left = 0.25, right = 0.25, top = 0.50, bottom = 0.50)
ws.repeat_rows(0)
ws.fit_to_pages(1, 3)

writer.close()

#Security Cheat Sheet - format and add non-public sensitivity label
wb_s = xw.Book("output\Security_Cheat_Sheet_CSO.xlsx")

ws = wb_s.sheets[tbl]
a_range = ws.used_range.address 
ws[a_range].font.name = "Calibri"
ws[a_range].font.size = 10
ws[a_range].columns.autofit()
ws[a_range].api.HorizontalAlignment = xw.constants.HAlign.xlHAlignCenter
ws[a_range].api.VerticalAlignment = xw.constants.VAlign.xlVAlignCenter

ws.api.PageSetup.PrintArea = a_range

labelinfo = wb_s.api.SensitivityLabel.CreateLabelInfo()
labelinfo.AssignmentMethod = 2
labelinfo.Justification = 'init'
labelinfo.LabelId = "7212d102-6a13-4378-b0e2-fdaee0e76765"
wb_s.api.SensitivityLabel.SetLabel(labelinfo, labelinfo)

wb_s.save()
wb_s.close()

**MPP**

In [ ]:
writer = pd.ExcelWriter("output\Exhibit_A_MPP.xlsx", engine="xlsxwriter") #Exhibit A - xlsx excel object

wb = writer.book #MPP - workbook object

#Filter dataframe
tbl = "Exhibit_A_MPP"

dftE = df.loc[(df["COLLATERAL_TABLE"] == "Expanded") & (df["MPP_ASSET_TYPE"].notna())].copy()
dftL = df.loc[(df["COLLATERAL_TABLE"] == "Limited") & (df["MPP_ASSET_TYPE"].notna())].copy()
dftQ = df.loc[(df["COLLATERAL_TABLE"] == "QCD") & (df["MPP_ASSET_TYPE"].notna())].copy()
dftS = df.loc[(df["COLLATERAL_TABLE"] == "Securities") & (df["MPP_ASSET_TYPE"].notna())].copy()

dftE.loc[(dftE["MPP_ASSET_SUB_TYPE"] == "Not Agency-eligible") | (dftE["MPP_ASSET_TYPE"] == "Agricultural real estate") | (dftE["MPP_ASSET_TYPE"] == "Second mortgages on 1-4 family residential real property") \
| (dftE["MPP_ASSET_TYPE"] == "Home Equity Line of Credit (HELOC)") | (dftE["MPP_ASSET_TYPE"] == "Residential construction mortgages") | (dftE["MPP_ASSET_TYPE"] == "Multifamily construction mortgages") \
| (dftE["MPP_ASSET_TYPE"] == "Commercial construction mortgages") | (dftE["MPP_ASSET_TYPE"] == "Non-lead lender participation loans") | (dftE["MPP_ASSET_TYPE"] == "Operating loans (CFI only)") \
| (dftE["MPP_ASSET_TYPE"] == "Equipment loans (CFI only)") | (dftE["MPP_ASSET_TYPE"] == "Student loans"), "PROPOSED_LV_CL_A"] = "Not applicable"

dftE.loc[(dftE["MPP_ASSET_TYPE"] == "Other real estate-related property"), "PROPOSED_LV_CL_A"] = "Case-by-Case"

dftE.loc[(dftE["MPP_ASSET_SUB_TYPE"] == "Not Agency-eligible") | (dftE["MPP_ASSET_TYPE"] == "Agricultural real estate") | (dftE["MPP_ASSET_TYPE"] == "Second mortgages on 1-4 family residential real property") \
| (dftE["MPP_ASSET_TYPE"] == "Home Equity Line of Credit (HELOC)") | (dftE["MPP_ASSET_TYPE"] == "Residential construction mortgages") | (dftE["MPP_ASSET_TYPE"] == "Multifamily construction mortgages") \
| (dftE["MPP_ASSET_TYPE"] == "Commercial construction mortgages") | (dftE["MPP_ASSET_TYPE"] == "Non-lead lender participation loans") | (dftE["MPP_ASSET_TYPE"] == "Operating loans (CFI only)") \
| (dftE["MPP_ASSET_TYPE"] == "Equipment loans (CFI only)") | (dftE["MPP_ASSET_TYPE"] == "Student loans"), "PROPOSED_LV_CL_B"] = "Not applicable"

dftE.loc[(dftE["MPP_ASSET_TYPE"] == "Other real estate-related property"), "PROPOSED_LV_CL_B"] = "Case-by-Case"

dftL.loc[(dftL["MPP_ASSET_TYPE"] == "Multifamily construction mortgages") | (dftL["MPP_ASSET_TYPE"] == "Commercial construction mortgages"), "PROPOSED_LV_CL_A"] = 0

dftL.loc[(dftL["MPP_ASSET_TYPE"] == "Non-lead lender participation loans"), "PROPOSED_LV_CL_A"] = "Applicable asset LV less 10%"

dftL.loc[(dftL["MPP_ASSET_TYPE"] == "Other real estate-related property"), "PROPOSED_LV_CL_A"] = "Case-by-Case"

dftQ.loc[(dftQ["MPP_ASSET_TYPE"] == "Multifamily construction mortgages") | (dftQ["MPP_ASSET_TYPE"] == "Commercial construction mortgages"), "PROPOSED_LV_CL_A"] = 0

dftQ.loc[(dftQ["MPP_ASSET_TYPE"] == "Non-lead lender participation loans"), "PROPOSED_LV_CL_A"] = "Applicable asset LV less 10%"

dftQ.loc[(dftQ["MPP_ASSET_TYPE"] == "Other real estate-related property"), "PROPOSED_LV_CL_A"] = "Case-by-Case"

dftS.loc[(dftS["MPP_ASSET_TYPE"] == "Mutual Funds") | (dftS["MPP_ASSET_TYPE"] == "State and local government securities") | (dftS["MPP_ASSET_SUB_TYPE"].str.contains("PLMBS - HE")) \
| (dftS["MPP_ASSET_SUB_TYPE"].str.contains("PLMBS - Inverse floater")) | (dftS["MPP_ASSET_SUB_TYPE"].str.contains("PLMBS - SUP, MEZZ")), "PROPOSED_LV_CL_A"] = "Case-by-Case"

dftS.loc[(dftS["MPP_ASSET_TYPE"] == "Mutual Funds") | (dftS["MPP_ASSET_TYPE"] == "State and local government securities") | (dftS["MPP_ASSET_SUB_TYPE"].str.contains("PLMBS - HE")) \
| (dftS["MPP_ASSET_SUB_TYPE"].str.contains("PLMBS - Inverse floater")) | (dftS["MPP_ASSET_SUB_TYPE"].str.contains("PLMBS - SUP, MEZZ")), "PROPOSED_LV_CL_B"] = "Case-by-Case"

dftS.loc[(dftS["MPP_ASSET_TYPE"] == "Commercial Mortgage-Backed Securities - 100% defeased (rated at least equivalent to the U.S. Government)") \
| (dftS["MPP_ASSET_TYPE"] == "State and local government securities"), "FHLB_RATING"] = "NR"

dftE["DELIVERED_EXPANDED_LV_CL_A"] = dftE["PROPOSED_LV_CL_A"] #reassign col
dftE["DELIVERED_EXPANDED_LV_CL_B"] = dftE["PROPOSED_LV_CL_B"] #reassign col

dftL["DELIVERED_LIMITED_LV"] = dftL["PROPOSED_LV_CL_A"] #reassign col

dftQ["BLANKET_QCD_LV"] = dftQ["PROPOSED_LV_CL_A"] #reassign col

dftS["BLANKET_QCD_LV"] = "Not applicable" #reassign col
dftS["DELIVERED_LIMITED_LV"] = dftS["PROPOSED_LV_CL_A"] #reassign col
dftS["DELIVERED_EXPANDED_LV_CL_A"] = dftS["PROPOSED_LV_CL_A"] #reassign col
dftS["DELIVERED_EXPANDED_LV_CL_B"] = dftS["PROPOSED_LV_CL_B"] #reassign col

#Merge loan dataframes
dftLN = dftL.merge(dftQ[["MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE", "BLANKET_QCD_LV"]], on=["MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE"], how="left") #ltd and qcd
dftLN = dftLN.merge(dftE[["MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE", "DELIVERED_EXPANDED_LV_CL_A", "DELIVERED_EXPANDED_LV_CL_B"]], on=["MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE"], how="left") #ltd, qcd and exp

cols = ["MPP_SORT", "MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE", "FHLB_RATING", "BLANKET_QCD_LV", "DELIVERED_LIMITED_LV", "DELIVERED_EXPANDED_LV_CL_A", "DELIVERED_EXPANDED_LV_CL_B"]

#Keep only MPP columns
dftLN = dftLN[cols]
dftS = dftS[cols]

#Stack dataframes
dft = pd.concat([dftLN, dftS], ignore_index=True, axis=0)

#Sort dataframe in MPP order
dft = dft.sort_values(["MPP_SORT", "MPP_ASSET_SUB_TYPE", "FHLB_RATING"], ascending=[True, True, False])

#Add rows; set index (for grouped excel output)
dft = add_rows(dft, "MPP_ASSET_TYPE")
dft = dft.set_index(["MPP_ASSET_TYPE", "MPP_ASSET_SUB_TYPE", "FHLB_RATING"])

#Create MPP sheet
cols = ["BLANKET_QCD_LV", "DELIVERED_LIMITED_LV", "DELIVERED_EXPANDED_LV_CL_A", "DELIVERED_EXPANDED_LV_CL_B"]

dft[cols].to_excel(writer, sheet_name = tbl, index=True) 
ws = writer.sheets[tbl] #worksheet object
ws.set_column("A:G", None, wb.add_format(WSFORMAT)) #set column format
ws.set_default_row(15) #set row height
ws.set_header('&L&A') #set header
wb.add_format(HEADFORMAT) #set header format

ws.set_paper(1)
ws.set_landscape()
ws.center_horizontally()
ws.set_margins(left = 0.25, right = 0.25, top = 0.50, bottom = 0.50)
ws.repeat_rows(0)
ws.fit_to_pages(1, 2)

writer.close()

#Exhibit A_MPP - format and add non-public sensitivity label
wb_mpp = xw.Book("output\Exhibit_A_MPP.xlsx")

ws = wb_mpp.sheets[tbl]
a_range = ws.used_range.address 
ws[a_range].font.name = "Calibri"
ws[a_range].font.size = 10
ws["A:B"].api.WrapText = True
ws["A:A"].column_width = 80
ws["B:B"].column_width = 48
ws["C:G"].columns.autofit()
ws[a_range].rows.autofit()
ws[a_range].api.HorizontalAlignment = xw.constants.HAlign.xlHAlignCenter
ws[a_range].api.VerticalAlignment = xw.constants.VAlign.xlVAlignCenter

ws.api.PageSetup.PrintArea = a_range

labelinfo = wb_mpp.api.SensitivityLabel.CreateLabelInfo()
labelinfo.AssignmentMethod = 2
labelinfo.Justification = 'init'
labelinfo.LabelId = "7212d102-6a13-4378-b0e2-fdaee0e76765"
wb_mpp.api.SensitivityLabel.SetLabel(labelinfo, labelinfo)

wb_mpp.save()
wb_mpp.close()